# Resolução 04 — Visão computacional aplicada ao meio ambiente

**Curso:** Tecnologia, Dados e IA na Gestão Ambiental
**Encontro:** 9 (opcional)

Este notebook resolvedor classifica **duas imagens sintéticas** com um modelo pré-treinado e
mostra, na prática, por que um classificador genérico **não serve** como laudo ambiental.

Ele roda com **Runtime → Run all** e baixa as imagens do repositório. Não precisa enviar
arquivo nem treinar nada.

> As imagens são **ilustrações sintéticas** geradas para a aula. Não são fotos de satélite nem
> representam um município real.

## Passo 0 — Baixar as imagens de exemplo

O código abaixo baixa duas imagens do próprio repositório: uma cena verde e uma cena
queimada. Se você quiser testar sua própria foto, use a opção de upload no fim.

In [ ]:
from pathlib import Path
import urllib.request

BASE_IMAGENS = "https://raw.githubusercontent.com/Marquezbertin/Curso-Tecnologia-Dados-Gestao-Ambiental/main/11-RESOLUCAO-DOS-EXERCICIOS"

def baixar(nome, base=BASE_IMAGENS):
    destino = Path(nome)
    if not destino.exists():
        print(f"Baixando {nome}...")
        urllib.request.urlretrieve(f"{base}/{nome}", destino)
    print(f"Imagem pronta: {destino.resolve()}")
    return destino

ARQUIVOS = [
    baixar("exemplo-area-verde.jpg"),
    baixar("exemplo-area-queimada.jpg"),
]

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, caminho in zip(axes, ARQUIVOS):
    ax.imshow(Image.open(caminho))
    ax.set_title(caminho.name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Passo 1 — Carregar o MobileNetV2

Usamos o MobileNetV2 do `torchvision`, que já vem no Colab, treinado no **ImageNet** (1.000
classes de objetos do dia a dia). O modelo **não foi treinado** para satélite, floresta,
queimada ou uso do solo.

In [ ]:
try:
    import torch
    from torchvision import transforms
    from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
except ImportError as erro:
    raise SystemExit(
        "PyTorch/torchvision não encontrado. No Colab, ele já vem instalado. "
        "Localmente, rode: pip install torch torchvision"
    ) from erro

print("PyTorch:", torch.__version__)

pesos = MobileNet_V2_Weights.IMAGENET1K_V1
modelo = mobilenet_v2(weights=pesos).eval()
preprocessamento = pesos.transforms()
categorias = pesos.meta["categories"]

print("Modelo carregado: MobileNetV2 / ImageNet")

## Passo 2 — Classificar as duas imagens

O modelo devolve as 5 classes mais prováveis para cada imagem. Repare que ele não retorna
"floresta" ou "área queimada": retorna objetos do ImageNet.

In [ ]:
def classificar(caminho, top=5):
    imagem = Image.open(caminho).convert("RGB")
    tensor = preprocessamento(imagem).unsqueeze(0)

    with torch.no_grad():
        probabilidades = modelo(tensor).softmax(dim=1)[0]

    valores, indices = torch.topk(probabilidades, top)
    return [(categorias[i], float(v)) for v, i in zip(valores, indices)]

resultados = {}

for caminho in ARQUIVOS:
    resultados[caminho.name] = classificar(caminho)
    print(f"\n{caminho.name}")
    print("-" * 55)
    for nome, prob in resultados[caminho.name]:
        print(f"{nome:30s} {prob:.3f}")

## Passo 3 — Comparar em tabela

A tabela organiza o resultado. Se as classes não tiverem relação com meio ambiente, esse é o
dado mais importante da aula.

In [ ]:
import pandas as pd

linhas = []
for arquivo, previsoes in resultados.items():
    for posicao, (nome, prob) in enumerate(previsoes, start=1):
        linhas.append({
            "imagem": arquivo,
            "posicao": posicao,
            "classe_imagenet": nome,
            "probabilidade": round(prob, 3),
        })

tabela = pd.DataFrame(linhas)
tabela

### Resposta observada (referência)

Com as imagens sintéticas deste repositório, o MobileNetV2 **não reconhece paisagem**. Em um
teste de referência, as saídas foram parecidas com:

**Área verde:**

| Classe ImageNet | Probabilidade |
|---|---:|
| balloon | 0,242 |
| envelope | 0,221 |
| street sign | 0,078 |
| web site | 0,016 |
| cliff | 0,016 |

**Área queimada:**

| Classe ImageNet | Probabilidade |
|---|---:|
| iPod | 0,279 |
| toilet tissue | 0,236 |
| paper towel | 0,145 |
| envelope | 0,073 |
| Band Aid | 0,067 |

Os valores exatos **podem variar** com a versão do torchvision e do modelo. O importante é a
conclusão: **nenhuma classe é ambiental**, e as probabilidades são baixas ou moderadas. O
modelo está tentando encaixar texturas e formas em objetos que ele aprendeu a reconhecer.

### Por que ele erra?

1. **Domínio diferente:** ImageNet tem fotos de objetos vistos de frente, não imagens aéreas.
2. **Sem treino local:** o modelo nunca viu a vegetação, o solo ou a fumaça da sua região.
3. **Textura e cor não bastam:** ele pode confundir fumaça com nuvem, pasto com floresta,
   telhado com água.
4. **Probabilidade não é certeza:** 0,30 não significa "30% de chance de ser floresta";
   significa confiança do modelo dentro das classes dele.

## Ficha de modelo preenchida

| Campo | Resposta |
|---|---|
| Modelo usado | MobileNetV2 pré-treinado no ImageNet (torchvision) |
| Treinado para reconhecer | 1.000 classes de objetos cotidianos |
| Imagens testadas | Duas ilustrações sintéticas: área verde e área queimada |
| Resultado (3 primeiras classes) | balloon/envelope/street sign e iPod/toilet tissue/paper towel |
| O modelo acertou? | Não. Nenhuma classe corresponde a paisagem, floresta ou queimada |
| Onde ele provavelmente erraria | Imagens aéreas, fumaça, dossel, solo exposto, áreas vistas de cima |
| Limitação para uso ambiental | Domínio de treino inadequado, sem validação local, sem rótulos ambientais |
| Uso recomendado | Demonstrar conceito de classificação e discutir erro e viés |
| Uso não recomendado | Laudo, alerta de desmatamento, fiscalização ou decisão automática |

### O que seria necessário para usar de verdade

- treinar com imagens de satélite da região, com rótulos feitos por especialistas;
- separar treino, validação e teste por **área geográfica** e por **ano**;
- medir acurácia, recall, precisão e matriz de confusão por classe;
- testar em imagens de outra estação e de outro município;
- documentar versão do modelo, dados, limitações e responsável;
- manter **revisão humana** antes de qualquer ação.

## Passo 4 — Ética e segurança

- Não envie imagens com pessoas identificáveis.
- Não use o resultado como laudo ambiental.
- Registre modelo, imagem testada e limitação.
- A decisão final é sempre humana e precisa citar a fonte.
- Se o modelo errar, o erro precisa ser documentado, não escondido.

### Perguntas com respostas

1. **O modelo aprendeu o conceito de floresta?** Não. Ele aprendeu padrões visuais de objetos
   do ImageNet. A classificação de paisagem exige outro conjunto de dados e outro problema.
2. **Probabilidade alta significa certeza?** Não. Significa confiança relativa dentro das
   classes treinadas. Um modelo pode estar confiante e errado.
3. **Quem é afetado por um falso negativo?** A comunidade que deveria receber alerta, a equipe
   de fiscalização e o ecossistema.
4. **O que fazer antes de publicar um resultado?** Conferir a imagem, a fonte, o modelo, a
   classe prevista e a limitação; pedir revisão humana.

## Opcional — Testar sua própria imagem

No Colab, você pode enviar uma foto do pátio, de uma árvore ou de uma área queimada. **Não
envie foto com rosto de pessoa.** Troque `USAR_UPLOAD` para `True` e rode a célula.

In [ ]:
USAR_UPLOAD = False  # troque para True no Colab se quiser enviar uma foto

if USAR_UPLOAD:
    from google.colab import files
    enviados = files.upload()
    ARQUIVOS_USUARIO = [Path(nome) for nome in enviados.keys()]
else:
    ARQUIVOS_USUARIO = ARQUIVOS

for caminho in ARQUIVOS_USUARIO:
    print(f"\n{caminho.name}")
    for nome, prob in classificar(caminho):
        print(f"{nome:30s} {prob:.3f}")

## Checklist de entrega

- [ ] O notebook rodou inteiro com **Run all**.
- [ ] As duas imagens foram classificadas.
- [ ] A tabela de resultados foi interpretada.
- [ ] A ficha de modelo está preenchida.
- [ ] O texto explica por que o modelo erra.
- [ ] O texto não trata a probabilidade como certeza.
- [ ] A limitação para uso ambiental está escrita.

**Voltar ao índice:** [`README.md`](README.md)